# 04. LightGBM + SHAP

## 트리 모델
"할인율 > 7%이고 프로모션이 있으면 → 판매 +300" 같은 **조건 규칙(의사결정나무)**을 수백 개 쌓아서 예측합니다(그래디언트 부스팅).
- 장점: 요인 간 **상호작용**("공휴일에 프로모션하면 효과가 더 크다")과 복잡한 곡선을 자동으로 찾습니다.
- 단점: 공식이 없어서 그 자체로는 설명이 안 되고, 데이터가 적으면 **과적합**하기 쉽습니다.

## SHAP으로 설명하기
SHAP은 각 예측을 **"평균 예측 + 요인별 기여"**로 쪼개 줍니다. 그래서 트리 모델도 주별 기여도 분해가 가능합니다.

> ⚠️ 차이점: 선형 회귀의 기여도는 **"그 요인이 없을 때" 대비**, SHAP은 **"평균적인 주" 대비**입니다.
> 그래서 SHAP에서는 프로모션이 없는 주의 프로모션 기여가 **마이너스**로 나옵니다(평균보다 프로모션이 적으니까).

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 글꼴
from common import *
plt.rcParams["figure.dpi"] = 110
df = load_data()
y = df.sellout.values.astype(float)
print(df.shape)

In [ ]:
t = np.arange(len(df), dtype=float)
ad = df.ad_spend.values.astype(float)
lag = lambda x, k: np.concatenate([np.zeros(k), x[:-k]])
F = pd.DataFrame(dict(
    t=t, s1=np.sin(2*np.pi*t/52), c1=np.cos(2*np.pi*t/52),
    disc=df.discount_pct.values, promo=df.promo.values,
    ad=ad, ad_l1=lag(ad, 1), ad_l2=lag(ad, 2), ad_l3=lag(ad, 3),   # 광고 이월은 지난 1~3주 값으로 스스로 배움
    hol=df.holiday.values, temp=df.temp.values, stock=df.stockout_days.values))
GROUP = {"t": "기본 수요", "s1": "계절성", "c1": "계절성", "disc": "할인", "promo": "프로모션",
         "ad": "광고", "ad_l1": "광고", "ad_l2": "광고", "ad_l3": "광고",
         "hol": "공휴일", "temp": "기온", "stock": "결품"}

def shap_to_contrib(shap):
    contrib = {f: np.zeros(len(df)) for f in FACTORS}
    contrib["기본 수요"] += shap[:, -1]          # 마지막 열 = 평균 예측(기준값)
    for j, c in enumerate(F.columns):
        contrib[GROUP[c]] += shap[:, j]
    return contrib
F.head()

In [ ]:
import lightgbm as lgb

def make_model():
    return lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=7, min_child_samples=5,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                             random_state=7, verbose=-1)

model = make_model().fit(F, y)
pred = model.predict(F)
print(f"학습 기간: R² = {r2(y, pred):.3f},  MAPE = {mape(y, pred):.1f}%")

## 과적합 확인: 새 데이터를 맞힐 수 있나?

학습 기간 오차가 아주 작다고 좋은 모델은 아닙니다. **모델이 보지 못한 미래 주**를 맞혀봐야 합니다(시계열 교차검증).

In [ ]:
FOLD_STARTS, FOLD_LEN = [72, 80, 88, 96], 8   # 앞부분으로 학습 → 다음 8주 예측, 4번 반복

def time_series_cv(fit_predict):
    """fit_predict(train_idx) → 전체 기간 예측. 모델이 보지 못한 다음 8주의 MAPE 평균"""
    errs = []
    for s0 in FOLD_STARTS:
        tr, te = np.arange(s0), np.arange(s0, min(s0 + FOLD_LEN, len(df)))
        errs.append(mape(y[te], fit_predict(tr)[te]))
    return np.mean(errs), np.round(errs, 1)

In [ ]:
cv_mean, cv_folds = time_series_cv(lambda tr: make_model().fit(F.iloc[tr], y[tr]).predict(F))
print(f"학습 기간 MAPE {mape(y, pred):.1f}%  vs  새 데이터 MAPE {cv_mean:.1f}% {cv_folds}")
print(f"→ 새 데이터 오차가 {cv_mean / mape(y, pred):.0f}배: 이미 본 데이터를 외웠을 뿐 규칙을 배우지 못함 (과적합)")

## 트리 모델은 추세를 이어가지 못한다 (외삽 불가)

트리는 학습 때 본 값의 범위 안에서만 답을 냅니다. 판매가 꾸준히 늘어나는 추세라면 미래를 **낮게** 예측합니다. 새 데이터 오차의 주원인입니다.

In [ ]:
tr = np.arange(80)
p_tree = make_model().fit(F.iloc[tr], y[tr]).predict(F)
M_ = np.column_stack([np.ones(len(df)), t])
b = np.linalg.lstsq(M_[tr], y[tr], rcond=None)[0]
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(df.week, y, color="black", lw=1, label="실제")
ax.plot(df.week, p_tree, color="#5a9a2f", label="LightGBM (앞 80주로 학습)")
ax.plot(df.week, M_ @ b, color="grey", ls="--", label="직선 추세")
ax.axvspan(df.week[80], df.week.iloc[-1], color="grey", alpha=.12, label="학습에 안 쓴 구간")
ax.legend(fontsize=8); ax.set_title("트리 모델의 미래 예측");

## SHAP 기여도 분해

In [ ]:
shap = model.predict(F, pred_contrib=True)   # LightGBM 내장 TreeSHAP
contrib = shap_to_contrib(shap)
print("SHAP 기준값(2년 평균 예측):", round(shap[0, -1], 1))
i = 29   # 프로모션 주 예시
plot_waterfall(contrib, i, actual=y[i], title=f"{df.week[i]:%Y-%m-%d} 주 (SHAP: '기본 수요' = 평균 판매)");

In [ ]:
plot_stacked(df, contrib, "주별 SHAP 기여도 (LightGBM)")
effects_table({"LightGBM": effects(contrib, df)}, df)

## 결과 해석
- **과적합**: 학습 기간 오차 0.9%, 새 데이터 오차 약 6%.
- **드문 이벤트를 약하게 배움**: 공휴일은 2년에 14주뿐이라 효과를 정답(180)보다 훨씬 작게 잡았습니다.
- **기온 효과를 거의 놓침**: 기온 효과가 계절 패턴(sin/cos) 쪽으로 흡수되었습니다.

**트리 모델이 유리해지는 조건**: 매장별·지역별 데이터로 수천 행 이상이 되고, 요인 간 상호작용이 실제로 있을 때.